In [1]:
# ============================================================================
# Imports
# ============================================================================

import gc
import json
import numpy as np
from rouge_score import rouge_scorer as rouge_scoring
from types import SimpleNamespace
from verbatim_rag.index import VerbatimIndex
from verbatim_rag.vector_stores import LocalMilvusStore
from verbatim_rag.embedding_providers import SentenceTransformersProvider
from verbatim_rag.extractors import LLMSpanExtractor, SemanticHighlightExtractor
from verbatim_rag.core import VerbatimRAG

In [2]:
# ============================================================================
# CELL 2: Lade Dev Data
# ============================================================================

def load_jsonl(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

dev_answerable = load_jsonl('../../data/clapnq_dev_answerable.jsonl')
dev_unanswerable = load_jsonl('../../data/clapnq_dev_unanswerable.jsonl')

print(f"Answerable: {len(dev_answerable)}")
print(f"Unanswerable: {len(dev_unanswerable)}")

Answerable: 300
Unanswerable: 300


In [3]:
# ============================================================================
# CELL 3: Lade Index (clapnq_milvus.db)
# ============================================================================

db_path = "../../data/clapnq_milvus_e5_v4.db"

dense_provider = SentenceTransformersProvider(
    model_name="intfloat/e5-base-v2",
    device="cpu"
)

vector_store = LocalMilvusStore(
    db_path=db_path,
    collection_name="clapnq_passages_e5",
    enable_dense=True,
    enable_sparse=False,
    dense_dim=768,
)

index = VerbatimIndex(
    vector_store=vector_store,
    dense_provider=dense_provider,
    chunker_provider=None,
)

print("✅ Index loaded!")

# Quick Test
results = index.query("what is the story of call of duty zombie", k=3)
print(f"Test query returned {len(results)} results")

/home/pkunerth/projects/verbatim-rag/.venv/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


✅ Index loaded!
Test query returned 3 results


In [4]:
# ============================================================================
# CELL 4: Helper Funktionen
# ============================================================================

def is_unanswerable_response(answer):
    """Prüfe ob die Antwort 'I don't know' bedeutet."""
    if not answer:
        return True
    
    unanswerable_phrases = [
        "i don't know",
        "i do not know", 
        "cannot be answered",
        "not answerable",
        "no answer",
        "unanswerable",
        "cannot answer",
        "not enough information",
        "insufficient information",
        "no relevant information found",
        "no relevant information",
        "not found in the provided",
        "cannot be determined",
        "does not contain",
        "no information",
    ]
    
    answer_lower = answer.lower().strip()
    return any(phrase in answer_lower for phrase in unanswerable_phrases)

def compute_rougeLp(generated_answer, passage_text):
    """RougeLp: Faithfulness zum Passage."""
    scorer = rouge_scoring.RougeScorer(['rougeL'], use_stemmer=False)
    scores = scorer.score(passage_text, generated_answer)
    return scores['rougeL'].precision

In [5]:
# ============================================================================
# Full RAG Pipeline - Run Funktion
# ============================================================================

# VerbatimRAG einmal erstellen, nicht pro Aufruf
_shared_rag = VerbatimRAG(index=None)

def run_full_rag_single(example, index, extractor, k=3):
    """
    Full RAG Pipeline für ein Beispiel.
    
    :param example: Ein Beispiel aus dem Dataset
    :param index: VerbatimIndex für Retrieval
    :param extractor: SpanExtractor-Instanz (LLM oder Semantic)
    :param k: Anzahl Top-K Passages
    :return: (system_answer, retrieved_results)
    """
    question = example['input']
    
    # Retrieve
    retrieved_results = index.query(text="query: " + question, k=k)
    
    # Extract
    search_results = [
        SimpleNamespace(text=result.text) 
        for result in retrieved_results
    ]
    relevant_spans = extractor.extract_spans(question, search_results)
    
    # Keine relevanten Spans → unanswerable
    if not relevant_spans or all(len(v) == 0 for v in relevant_spans.values()):
        return "", retrieved_results
    
    # Generate
    display_spans, citation_spans = _shared_rag._rank_and_split_spans(relevant_spans)
    system_answer = _shared_rag.template_manager.process(
        question, display_spans, citation_spans
    )
    
    return system_answer, retrieved_results

from verbatim_core.templates import TemplateManager

tm_spans_only = TemplateManager(default_mode="static")
tm_spans_only.use_static_mode(template="[DISPLAY_SPANS]")
tm_spans_only.set_citation_mode("hidden")
_shared_rag_spans_only = VerbatimRAG(index=None, template_manager=tm_spans_only)

def run_full_rag_single_spans_only(example, index, extractor, k=3):
    question = example['input']
    retrieved_results = index.query(text="query: " + question, k=k)
    search_results = [SimpleNamespace(text=r.text) for r in retrieved_results]
    relevant_spans = extractor.extract_spans(question, search_results)
    if not relevant_spans or all(len(v) == 0 for v in relevant_spans.values()):
        return "", retrieved_results
    display_spans, citation_spans = _shared_rag_spans_only._rank_and_split_spans(relevant_spans)
    system_answer = _shared_rag_spans_only.template_manager.process(
        question, display_spans, citation_spans
    )
    return system_answer, retrieved_results

In [6]:
# ============================================================================
# Batch Evaluation mit Memory Cleanup + Interim Saves
# ============================================================================

def evaluate_in_batches(
    data, index, extractor, k=3, batch_size=50,
    data_type="answerable", label="system", run_fn=None
):
    """Evaluate in batches with memory cleanup and intermediate saves."""
    run_fn = run_fn or run_full_rag_single
    scorer = rouge_scoring.RougeScorer(['rougeL', 'rouge1'], use_stemmer=False)

    all_rougeL, all_recall, all_rougeLp, all_length, all_unans = [], [], [], [], []
    all_outputs = [] 

    for batch_start in range(0, len(data), batch_size):
        batch_end = min(batch_start + batch_size, len(data))
        batch = data[batch_start:batch_end]
        print(f"\n--- Batch {batch_start}-{batch_end} / {len(data)} ({data_type}) ---")

        for i, example in enumerate(batch):
            try:
                system_answer, retrieved_results = run_fn(
                    example, index, extractor=extractor, k=k
                )
            except Exception as e:
                print(f"Error on example {batch_start + i}: {e}")
                system_answer = ""
                retrieved_results = []

            if data_type == "answerable":
                reference_answer = example['output'][0]['answer']
                if system_answer:
                    scores = scorer.score(reference_answer, system_answer)
                    all_rougeL.append(scores['rougeL'].fmeasure * 100)
                    all_recall.append(scores['rouge1'].recall * 100)
                    all_retrieved_text = " ".join(r.text for r in retrieved_results)
                    all_rougeLp.append(compute_rougeLp(system_answer, all_retrieved_text) * 100)
                    all_length.append(len(system_answer))
                else:
                    all_rougeL.append(0.0)
                    all_recall.append(0.0)
                    all_rougeLp.append(0.0)
                    all_length.append(0)

                all_outputs.append({
                        "question_id": example.get("id", ""),
                        "question":    example["input"],
                        "answer":      system_answer,
                        "reference":   reference_answer,
                    })
            else:
                all_unans.append(1.0 if is_unanswerable_response(system_answer) else 0.0)

        # Memory cleanup after each batch
        gc.collect()

        # Save intermediate results
        if data_type == "answerable":
            interim = {
                'rougeL': float(np.mean(all_rougeL)),
                'recall': float(np.mean(all_recall)),
                'rougeLp': float(np.mean(all_rougeLp)),
                'avg_len': float(np.mean(all_length)),
                'n_done': batch_end,
            }
        else:
            interim = {'unans_acc': float(np.mean(all_unans) * 100), 'n_done': batch_end}

        with open(f'{label}_{data_type}_interim.json', 'w') as f:
            json.dump(interim, f, indent=2)
        print(f"  Saved interim. ({batch_end}/{len(data)})")

        if data_type == "answerable" and all_outputs:
            output_path = f"{label}_outputs.jsonl"
            with open(output_path, "w", encoding="utf-8") as f:
                for item in all_outputs:
                    f.write(json.dumps(item, ensure_ascii=False) + "\n")
            print(f"  Outputs gespeichert: {output_path}")

    if data_type == "answerable":
        return {
            'rougeL': float(np.mean(all_rougeL)),
            'recall': float(np.mean(all_recall)),
            'rougeLp': float(np.mean(all_rougeLp)),
            'avg_len': int(np.mean(all_length)) if all_length else 0,
        }
    else:
        return {'unans_acc': float(np.mean(all_unans) * 100)}


def evaluate_full_rag(
    dev_answerable,
    dev_unanswerable,
    index,
    extractor,
    k=3,
    n_samples=None,
    batch_size=50,
    label="System",
    run_fn=None
):
    """
    Evaluiert einen Extractor auf der Full-RAG Pipeline.
    Nutzt Batch-Evaluation mit gc.collect() und Interim-Saves.
    """
    if n_samples:
        answerable_data = dev_answerable[:n_samples]
        unanswerable_data = dev_unanswerable[:n_samples]
    else:
        answerable_data = dev_answerable
        unanswerable_data = dev_unanswerable

    print(f"\n{'='*60}")
    print(f"  {label} (Top-{k} RAG)")
    print(f"{'='*60}")

    # Answerable
    print(f"\nEvaluating {len(answerable_data)} ANSWERABLE examples...")
    ans_results = evaluate_in_batches(
        answerable_data, index, extractor, k=k,
        batch_size=batch_size, data_type="answerable", label=label, run_fn=run_fn
    )

    # Unanswerable
    print(f"\nEvaluating {len(unanswerable_data)} UNANSWERABLE examples...")
    unans_results = evaluate_in_batches(
        unanswerable_data, index, extractor, k=k,
        batch_size=batch_size, data_type="unanswerable", label=label, run_fn=run_fn
    )

    # Combined results
    results = {**ans_results, **unans_results}

    print(f"\n{'='*60}")
    print(f"{label} RESULTS (n={len(answerable_data)}):")
    print(f"  RougeL:     {results['rougeL']:.1f}")
    print(f"  Recall:     {results['recall']:.1f}")
    print(f"  RougeLp:    {results['rougeLp']:.1f}")
    print(f"  Avg Length: {results['avg_len']} chars")
    print(f"  Unanswerable Accuracy: {results['unans_acc']:.1f}%")
    print(f"{'='*60}")

    return results

In [7]:
llm_extractor = LLMSpanExtractor(span_match_mode="fuzzy")

In [8]:
# ============================================================================
# CELL 7: LLMSpanExtractor - Full RAG Evaluation
# ============================================================================

results_llm = evaluate_full_rag(
    dev_answerable,
    dev_unanswerable,
    index,
    extractor=llm_extractor,
    k=3,
    n_samples=None,
    label="LLMSpanExtractor"
)


  LLMSpanExtractor (Top-3 RAG)

Evaluating 300 ANSWERABLE examples...

--- Batch 0-50 / 300 (answerable) ---


  Saved interim. (50/300)
  Outputs gespeichert: LLMSpanExtractor_outputs.jsonl

--- Batch 50-100 / 300 (answerable) ---


  Saved interim. (100/300)
  Outputs gespeichert: LLMSpanExtractor_outputs.jsonl

--- Batch 100-150 / 300 (answerable) ---
  Saved interim. (150/300)
  Outputs gespeichert: LLMSpanExtractor_outputs.jsonl

--- Batch 150-200 / 300 (answerable) ---


  Saved interim. (200/300)
  Outputs gespeichert: LLMSpanExtractor_outputs.jsonl

--- Batch 200-250 / 300 (answerable) ---


  Saved interim. (250/300)
  Outputs gespeichert: LLMSpanExtractor_outputs.jsonl

--- Batch 250-300 / 300 (answerable) ---
  Saved interim. (300/300)
  Outputs gespeichert: LLMSpanExtractor_outputs.jsonl

Evaluating 300 UNANSWERABLE examples...

--- Batch 0-50 / 300 (unanswerable) ---


  Saved interim. (50/300)

--- Batch 50-100 / 300 (unanswerable) ---
  Saved interim. (100/300)

--- Batch 100-150 / 300 (unanswerable) ---


  Saved interim. (150/300)

--- Batch 150-200 / 300 (unanswerable) ---
  Saved interim. (200/300)

--- Batch 200-250 / 300 (unanswerable) ---


  Saved interim. (250/300)

--- Batch 250-300 / 300 (unanswerable) ---
  Saved interim. (300/300)

LLMSpanExtractor RESULTS (n=300):
  RougeL:     32.8
  Recall:     65.2
  RougeLp:    69.9
  Avg Length: 628 chars
  Unanswerable Accuracy: 22.3%


In [9]:
import os
import json
os.makedirs('full_rag_results', exist_ok=True)

with open('full_rag_results/results_llm.json', 'w') as f:
    json.dump(results_llm, f, indent=2)

In [10]:
# ============================================================================
# CELL 8: SemanticHighlightExtractor - Full RAG Evaluation
# ============================================================================

semantic_extractor = SemanticHighlightExtractor(
    threshold=0.5,
    output_mode="sentences",
)

results_semantic = evaluate_full_rag(
    dev_answerable,
    dev_unanswerable,
    index,
    extractor=semantic_extractor,
    k=3,
    n_samples=None,
    label="SemanticHighlight"
)


  SemanticHighlight (Top-3 RAG)

Evaluating 300 ANSWERABLE examples...

--- Batch 0-50 / 300 (answerable) ---


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 49.05it/s]


[OpenProvenceModel] Model inference time: 2.17s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 692.24it/s]


[OpenProvenceModel] Model inference time: 0.48s (1 blocks)


...

[OpenProvenceModel] Model inference time: 0.62s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 175.59it/s]


[OpenProvenceModel] Model inference time: 1.14s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 180.52it/s]


[OpenProvenceModel] Model inference time: 1.26s (1 blocks)
  Saved interim. (300/300)

SemanticHighlight RESULTS (n=300):
  RougeL:     27.0
  Recall:     54.7
  RougeLp:    63.8
  Avg Length: 585 chars
  Unanswerable Accuracy: 41.3%


In [11]:
with open('full_rag_results/results_semantic.json', 'w') as f:
    json.dump(results_semantic, f, indent=2)

In [12]:
# LLM Extractor – Spans Only
results_llm_spans_only = evaluate_full_rag(
    dev_answerable, dev_unanswerable, index,
    extractor=llm_extractor, k=3, n_samples=None,
    label="LLMSpanExtractor_spans_only",
    run_fn=run_full_rag_single_spans_only,
)


  LLMSpanExtractor_spans_only (Top-3 RAG)

Evaluating 300 ANSWERABLE examples...

--- Batch 0-50 / 300 (answerable) ---


  Saved interim. (50/300)
  Outputs gespeichert: LLMSpanExtractor_spans_only_outputs.jsonl

--- Batch 50-100 / 300 (answerable) ---


  Saved interim. (100/300)
  Outputs gespeichert: LLMSpanExtractor_spans_only_outputs.jsonl

--- Batch 100-150 / 300 (answerable) ---
  Saved interim. (150/300)
  Outputs gespeichert: LLMSpanExtractor_spans_only_outputs.jsonl

--- Batch 150-200 / 300 (answerable) ---


  Saved interim. (200/300)
  Outputs gespeichert: LLMSpanExtractor_spans_only_outputs.jsonl

--- Batch 200-250 / 300 (answerable) ---
  Saved interim. (250/300)
  Outputs gespeichert: LLMSpanExtractor_spans_only_outputs.jsonl

--- Batch 250-300 / 300 (answerable) ---
  Saved interim. (300/300)
  Outputs gespeichert: LLMSpanExtractor_spans_only_outputs.jsonl

Evaluating 300 UNANSWERABLE examples...

--- Batch 0-50 / 300 (unanswerable) ---
  Saved interim. (50/300)

--- Batch 50-100 / 300 (unanswerable) ---
  Saved interim. (100/300)

--- Batch 100-150 / 300 (unanswerable) ---
  Saved interim. (150/300)

--- Batch 150-200 / 300 (unanswerable) ---
  Saved interim. (200/300)

--- Batch 200-250 / 300 (unanswerable) ---


  Saved interim. (250/300)

--- Batch 250-300 / 300 (unanswerable) ---
  Saved interim. (300/300)

LLMSpanExtractor_spans_only RESULTS (n=300):
  RougeL:     39.4
  Recall:     60.2
  RougeLp:    95.5
  Avg Length: 448 chars
  Unanswerable Accuracy: 23.3%


In [13]:
with open('full_rag_results/results_llm_spans_only.json', 'w') as f:
    json.dump(results_llm_spans_only, f, indent=2)

In [14]:


# Semantic Extractor – Spans Only
results_semantic_spans_only = evaluate_full_rag(
    dev_answerable, dev_unanswerable, index,
    extractor=semantic_extractor, k=3, n_samples=None,
    label="SemanticHighlight_spans_only",
    run_fn=run_full_rag_single_spans_only,
)


  SemanticHighlight_spans_only (Top-3 RAG)

Evaluating 300 ANSWERABLE examples...

--- Batch 0-50 / 300 (answerable) ---


Prepare contexts:   0%|          | 0/1 [00:00<?, ?it/s]

Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 108.24it/s]


[OpenProvenceModel] Model inference time: 3.06s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 401.14it/s]


...

[OpenProvenceModel] Model inference time: 0.90s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 303.65it/s]


[OpenProvenceModel] Model inference time: 1.36s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 232.87it/s]


[OpenProvenceModel] Model inference time: 1.42s (1 blocks)
  Saved interim. (300/300)

SemanticHighlight_spans_only RESULTS (n=300):
  RougeL:     31.7
  Recall:     51.1
  RougeLp:    86.3
  Avg Length: 442 chars
  Unanswerable Accuracy: 41.3%


In [15]:
with open('full_rag_results/results_semantic_spans_only.json', 'w') as f:
    json.dump(results_semantic_spans_only, f, indent=2)

In [16]:
from tabulate import tabulate

with open('full_rag_results/results_llm.json') as f:
    results_llm = json.load(f)
with open('full_rag_results/results_llm_spans_only.json') as f:
    results_llm_spans_only = json.load(f)
with open('full_rag_results/results_semantic.json') as f:
    results_semantic = json.load(f)
with open('full_rag_results/results_semantic_spans_only.json') as f:
    results_semantic_spans_only = json.load(f)

# --- ab hier alles gleich lassen ---

paper_baselines = [
    ("E5 + GPT-3.5",             34.0, 52.8, 30.0, 459, 27.3),
    ("E5 + Mistral-7B-Instruct", 31.3, 49.4, 30.1, 436, 11.7),
    ("MiniLM + CLAPNQ-T5-LG",   36.6, 46.4, 52.6, 300, 49.8),
    ("BGE + CLAPNQ-T5-LG",      40.7, 52.3, 54.2, 331, 41.9),
    ("E5 + CLAPNQ-T5-LG",       42.8, 54.3, 53.8, 343, 40.1),
]

our_results = [
    ("E5 + VerbatimRAG-LLM (contextual)",
        results_llm['rougeL'], results_llm['recall'],
        results_llm['rougeLp'], results_llm['avg_len'], results_llm['unans_acc']),
    ("E5 + VerbatimRAG-LLM (spans only)",
        results_llm_spans_only['rougeL'], results_llm_spans_only['recall'],
        results_llm_spans_only['rougeLp'], results_llm_spans_only['avg_len'],
        results_llm_spans_only['unans_acc']),
    ("E5 + VerbatimRAG-Semantic (contextual)",
        results_semantic['rougeL'], results_semantic['recall'],
        results_semantic['rougeLp'], results_semantic['avg_len'], results_semantic['unans_acc']),
    ("E5 + VerbatimRAG-Semantic (spans only)",
        results_semantic_spans_only['rougeL'], results_semantic_spans_only['recall'],
        results_semantic_spans_only['rougeLp'], results_semantic_spans_only['avg_len'],
        results_semantic_spans_only['unans_acc']),
]

headers = ["Model", "RougeL", "Recall", "RougeLp", "Len", "Unans\\%"]
all_rows = paper_baselines + our_results
n_paper = len(paper_baselines)

latex = tabulate(all_rows, headers=headers, tablefmt="latex_booktabs", floatfmt=".1f")
lines, data_count, result = latex.split("\n"), 0, []
for line in lines:
    result.append(line)
    if "&" in line:
        data_count += 1
        if data_count == n_paper:
            result.append(r"\midrule")
print("\n".join(result))

print("\n--- Markdown ---")
print(tabulate(all_rows, headers=headers, tablefmt="pipe", floatfmt=".1f"))

\begin{tabular}{lrrrrr}
\toprule
 Model                                  &   RougeL &   Recall &   RougeLp &   Len &   Unans\textbackslash{}\% \\
\midrule
 E5 + GPT-3.5                           &     34.0 &     52.8 &      30.0 &   459 &      27.3 \\
 E5 + Mistral-7B-Instruct               &     31.3 &     49.4 &      30.1 &   436 &      11.7 \\
 MiniLM + CLAPNQ-T5-LG                  &     36.6 &     46.4 &      52.6 &   300 &      49.8 \\
 BGE + CLAPNQ-T5-LG                     &     40.7 &     52.3 &      54.2 &   331 &      41.9 \\
\midrule
 E5 + CLAPNQ-T5-LG                      &     42.8 &     54.3 &      53.8 &   343 &      40.1 \\
 E5 + VerbatimRAG-LLM (contextual)      &     32.8 &     65.2 &      69.9 &   628 &      22.3 \\
 E5 + VerbatimRAG-LLM (spans only)      &     39.4 &     60.2 &      95.5 &   448 &      23.3 \\
 E5 + VerbatimRAG-Semantic (contextual) &     27.0 &     54.7 &      63.8 &   585 &      41.3 \\
 E5 + VerbatimRAG-Semantic (spans only) &     31.7 &     51.